In [1]:
import sys, csv, os, re, time
import pandas as pd
sys.path.append("../src")
import rag

# (qid, tier, expected_behavior, question, expected_docs)
# expected_docs format: "Name|season;Name|season"
EVAL = [
    # Tier 1 — simple lookup
    (1, 1, "answer", "How many points per game did Nikola Jokić average in 2023-24?", "Nikola Jokić|2023-24"),
    (2, 1, "answer", "What was the Boston Celtics' win-loss record in 2023-24?", "Boston Celtics|2023-24"),
    (3, 1, "answer", "How many games did Joel Embiid play in 2023-24?", "Joel Embiid|2023-24"),
    (4, 1, "answer", "What was Anthony Edwards' three-point percentage in 2023-24?", "Anthony Edwards|2023-24"),
    (5, 1, "answer", "How many assists per game did Tyrese Haliburton average in 2023-24?", "Tyrese Haliburton|2023-24"),
    (6, 1, "answer", "What was the Memphis Grizzlies' record in 2024-25?", "Memphis Grizzlies|2024-25"),
    (7, 1, "answer", "How many rebounds per game did Domantas Sabonis average in 2023-24?", "Domantas Sabonis|2023-24"),
    (8, 1, "answer", "What was Shai Gilgeous-Alexander's field goal percentage in 2023-24?", "Shai Gilgeous-Alexander|2023-24"),
    # Tier 2 — filtered
    (9, 2, "answer", "How did Jayson Tatum perform in road games in 2023-24?", "Jayson Tatum|2023-24"),
    (10, 2, "answer", "What was the Cleveland Cavaliers' home record in 2024-25?", "Cleveland Cavaliers|2024-25"),
    (11, 2, "missing", "How did Stephen Curry shoot from three in wins in 2023-24?", "Stephen Curry|2023-24"),
    (12, 2, "aggregate", "Which Lakers player scored the most points in a single game in 2023-24?", ""),
    (13, 2, "missing", "How did Victor Wembanyama rebound in games the Spurs lost in 2023-24?", "Victor Wembanyama|2023-24"),
    (14, 2, "answer", "What was the Indiana Pacers' highest-scoring game of 2023-24?", "Indiana Pacers|2023-24"),
    (15, 2, "missing", "How many double-doubles did Giannis Antetokounmpo record in 2023-24?", "Giannis Antetokounmpo|2023-24"),
    (16, 2, "missing", "How did the Oklahoma City Thunder perform in the second half of the 2024-25 season?", "Oklahoma City Thunder|2024-25"),
    # Tier 3 — comparison
    (17, 3, "missing", "Compare Luka Dončić's scoring in 2023-24 versus 2024-25.", "Luka Dončić|2023-24"),
    (18, 3, "answer", "Who shot better from three in 2023-24, Stephen Curry or Anthony Edwards?", "Stephen Curry|2023-24;Anthony Edwards|2023-24"),
    (19, 3, "answer", "Compare Jalen Brunson's scoring in wins versus losses in 2023-24.", "Jalen Brunson|2023-24"),
    (20, 3, "answer", "How did the Boston Celtics' offense compare to the Denver Nuggets' in 2024-25?", "Boston Celtics|2024-25;Denver Nuggets|2024-25"),
    (21, 3, "refuse", "Which player improved most in scoring between 2023-24 and 2024-25?", ""),
    (22, 3, "answer", "Compare Rudy Gobert's rebounding to Domantas Sabonis's in 2023-24.", "Rudy Gobert|2023-24;Domantas Sabonis|2023-24"),
    (23, 3, "answer", "Did Devin Booker score more at home or on the road in 2023-24?", "Devin Booker|2023-24"),
    (24, 3, "missing", "Compare the New York Knicks' record in close games to their overall record in 2023-24.", "New York Knicks|2023-24"),
    # Tier 4 — multi-step / aggregate
    (25, 4, "aggregate", "Which players averaged over 25 points while shooting above 50% in 2023-24?", ""),
    (26, 4, "missing", "How did the Minnesota Timberwolves perform against teams with winning records in 2023-24?", "Minnesota Timberwolves|2023-24"),
    (27, 4, "aggregate", "Which players had the biggest gap between home and road scoring in 2023-24?", ""),
    (28, 4, "missing", "Did the Milwaukee Bucks win more often when Giannis Antetokounmpo scored 30 or more in 2023-24?", "Giannis Antetokounmpo|2023-24"),
    (29, 4, "aggregate", "Which team had the best point differential in 2024-25?", ""),
    (30, 4, "answer", "How did the Oklahoma City Thunder's point differential compare to their win total in 2024-25?", "Oklahoma City Thunder|2024-25"),
    # Tier 5 — out of scope
    (31, 5, "refuse", "What is Nikola Jokić's contract worth?", ""),
    (32, 5, "refuse", "Who will win the NBA championship next season?", ""),
    (33, 5, "refuse", "How did the Golden State Warriors perform in the 2016-17 season?", ""),
    (34, 5, "refuse", "What is Victor Wembanyama's wingspan?", ""),
    (35, 5, "refuse", "Which college did Paolo Banchero attend?", ""),
    (36, 5, "refuse", "How many playoff games did the Boston Celtics win in 2023-24?", ""),
    (37, 5, "refuse", "How many points per game did Luka Dončić average in 2024-25?", ""),
    # Tier 6 — subjective, should ground in stats
    (38, 6, "grounded", "Who was the best player in the league in 2023-24?", ""),
    (39, 6, "grounded", "Did the Atlanta Hawks have a good season in 2023-24?", "Atlanta Hawks|2023-24"),
    (40, 6, "grounded", "Is Chet Holmgren a good defender based on his 2023-24 numbers?", "Chet Holmgren|2023-24"),
]

os.makedirs("../eval", exist_ok=True)
with open("../eval/eval_set.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["qid", "tier", "behavior", "question", "expected_docs"])
    w.writerows(EVAL)

ev = pd.read_csv("../eval/eval_set.csv").fillna("")
print(ev["behavior"].value_counts().to_string())

W1102 20:16:12.588000 85863 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

behavior
answer       17
missing       8
refuse        8
aggregate     4
grounded      3


In [2]:
key_to_id = {}
for d in rag.docs:
    name = d.get("player") or d.get("team")
    key_to_id[(name.lower(), d["season"])] = d["id"]

def expected_ids(s):
    if not s:
        return []
    out = []
    for item in s.split(";"):
        name, season = item.split("|")
        out.append(key_to_id.get((name.strip().lower(), season.strip()), f"MISSING:{item}"))
    return out

ev["expected_ids"] = ev["expected_docs"].apply(expected_ids)
bad = ev[ev["expected_ids"].apply(lambda ids: any(i.startswith("MISSING") for i in ids))]
print(f"{len(bad)} questions reference documents not in the corpus")
if len(bad):
    print(bad[["qid", "question", "expected_docs"]].to_string(index=False))

0 questions reference documents not in the corpus


In [7]:
DECLINE = ["no information", "not provided", "not present", "not mention", "does not mention",
           "doesn't mention", "not contain", "doesn't contain", "does not contain",
           "does not include", "doesn't include", "no data", "not available",
           "cannot", "can't", "unable", "only covers", "only contains", "no documents"]

def numbers_in(text):
    text = re.sub(r"[Dd]ocument \d+", " ", text)        # ignore "Document 1"
    text = re.sub(r"20\d\d[-–]\d\d", " ", text)
    text = re.sub(r"\b(19|20)\d\d\b", " ", text)
    return set(re.findall(r"\d+(?:\.\d+)?", text))

rows, t0 = [], time.time()
for _, q in ev.iterrows():
    r = rag.answer(q["question"])
    exp = [i for i in q["expected_ids"] if not i.startswith("MISSING")]
    unsupported = sorted(numbers_in(r["answer"]) - numbers_in(r["context"]))
    rows.append({
        "qid": q["qid"], "tier": q["tier"], "behavior": q["behavior"],
        "question": q["question"], "answer": r["answer"],
        "retrieved": "; ".join(r["source_names"]),
        "expected_ids": q["expected_ids"],
        "recall_at_3": (sum(i in r["sources"] for i in exp) / len(exp)) if exp else None,
        "declined": any(k in r["answer"].lower() for k in DECLINE),
        "unsupported_numbers": ", ".join(unsupported),
        "has_unsupported": bool(unsupported),
    })
    print(f"[{q['qid']:>2}] {time.time()-t0:5.0f}s  {q['question'][:65]}")

res = pd.DataFrame(rows)
res.to_csv("../eval/results_baseline.csv", index=False)
print(f"\nDone in {time.time()-t0:.0f}s")

[ 1]     4s  How many points per game did Nikola Jokić average in 2023-24?
[ 2]     6s  What was the Boston Celtics' win-loss record in 2023-24?
[ 3]     7s  How many games did Joel Embiid play in 2023-24?
[ 4]     8s  What was Anthony Edwards' three-point percentage in 2023-24?
[ 5]    10s  How many assists per game did Tyrese Haliburton average in 2023-2
[ 6]    11s  What was the Memphis Grizzlies' record in 2024-25?
[ 7]    12s  How many rebounds per game did Domantas Sabonis average in 2023-2
[ 8]    14s  What was Shai Gilgeous-Alexander's field goal percentage in 2023-
[ 9]    15s  How did Jayson Tatum perform in road games in 2023-24?
[10]    17s  What was the Cleveland Cavaliers' home record in 2024-25?
[11]    20s  How did Stephen Curry shoot from three in wins in 2023-24?
[12]    21s  Which Lakers player scored the most points in a single game in 20
[13]    27s  How did Victor Wembanyama rebound in games the Spurs lost in 2023
[14]    29s  What was the Indiana Pacers' highest-

In [8]:
summary = res.groupby("behavior").agg(
    n=("qid", "size"),
    retrieval_recall=("recall_at_3", "mean"),
    declined=("declined", "mean"),
    unsupported_numbers=("has_unsupported", "mean"),
).round(2)
print(summary.to_string())

print(f"\nOverall: {res['has_unsupported'].mean():.0%} of answers contain a number "
      f"not found in the retrieved context")

            n  retrieval_recall  declined  unsupported_numbers
behavior                                                      
aggregate   4               NaN      0.00                 0.25
answer     17               1.0      0.00                 0.12
grounded    3               1.0      0.33                 0.33
missing     8               1.0      0.50                 0.25
refuse      8               NaN      0.88                 0.00

Overall: 15% of answers contain a number not found in the retrieved context


In [9]:
pd.set_option("display.max_colwidth", None)
flagged = res[res["behavior"].isin(["missing", "aggregate"])][
    ["qid", "behavior", "question", "answer", "unsupported_numbers", "declined"]
]
for _, r in flagged.iterrows():
    print(f"[{r['qid']}] {r['behavior'].upper()}  declined={r['declined']}")
    print(f"Q: {r['question']}")
    print(f"A: {r['answer']}")
    print(f"Unsupported: {r['unsupported_numbers'] or '(none)'}")
    print("-" * 72)

[11] MISSING  declined=False
Q: How did Stephen Curry shoot from three in wins in 2023-24?
A: Based on the statistics provided in Document 1, Stephen Curry shot 40.7% from three (11.8 attempts per game) in 2023-24. In wins (42 games), he averaged 26.5 points.
Unsupported: (none)
------------------------------------------------------------------------
[12] AGGREGATE  declined=False
Q: Which Lakers player scored the most points in a single game in 2023-24?
A: Based on the provided context, the answer is not directly available. The context mentions the Lakers' highest-scoring game as 150 points, but it does not specify which player scored the most points in that game.
Unsupported: (none)
------------------------------------------------------------------------
[13] MISSING  declined=True
Q: How did Victor Wembanyama rebound in games the Spurs lost in 2023-24?
A: Based on the given data, we can find the information for Victor Wembanyama's rebounds in games the Spurs lost. In Document 1, it 

In [10]:
print(res[(res["behavior"] == "refuse") & (~res["declined"])][
    ["qid", "question", "answer"]].to_string(index=False))

 qid                               question                                                 answer
  31 What is Nikola Jokić's contract worth? This information is missing from the provided context.


In [11]:
res = pd.read_csv("../eval/results_baseline.csv")
res["grade"] = ""
res[["qid", "tier", "behavior", "question", "answer", "retrieved",
     "unsupported_numbers", "grade"]].to_csv("../eval/grading_sheet.csv", index=False)

In [12]:
GRADES = {
    1: "correct", 2: "correct", 3: "correct", 4: "correct", 5: "correct",
    6: "correct", 7: "correct", 8: "correct", 9: "correct", 10: "correct",
    11: "substituted", 12: "correct_refusal", 13: "correct_refusal", 14: "correct",
    15: "correct_refusal", 16: "correct_refusal", 17: "correct_refusal", 18: "correct",
    19: "correct", 20: "correct", 21: "correct_refusal", 22: "correct",
    23: "wrong", 24: "correct_refusal", 25: "sample_as_population", 26: "fabricated",
    27: "sample_as_population", 28: "substituted", 29: "sample_as_population",
    30: "correct", 31: "correct_refusal", 32: "correct_refusal", 33: "correct_refusal",
    34: "correct_refusal", 35: "correct_refusal", 36: "correct_refusal",
    37: "correct_refusal", 38: "correct", 39: "correct", 40: "correct",
}

g = pd.read_csv("../eval/grading_sheet.csv")
g["grade"] = g["qid"].map(GRADES)
g.to_csv("../eval/grading_sheet.csv", index=False)

good = g["grade"].isin(["correct", "correct_refusal"])
print(f"Overall: {good.sum()}/{len(g)} correct ({good.mean():.0%})")
print()
print(pd.crosstab(g["behavior"], g["grade"]).to_string())

Overall: 33/40 correct (82%)

grade      correct  correct_refusal  fabricated  sample_as_population  substituted  wrong
behavior                                                                                 
aggregate        0                1           0                     3            0      0
answer          16                0           0                     0            0      1
grounded         3                0           0                     0            0      0
missing          0                5           1                     0            2      0
refuse           0                8           0                     0            0      0
